# Comparação de trajetórias — Odometria vs Dead Reckoning
Todas as séries são normalizadas para começar em (0, 0) com theta=0.
A rotação inicial é removida para alinhar as trajetórias corretamente.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

# ── Altere somente esta variável ────────────────────────────────────────
PASTA_DADOS = Path('/home/rafael/odo_vs_dc/resultados/lar_20260504_014125')
# ────────────────────────────────────────────────────────────────────────

plt.rcParams.update({
    'figure.dpi': 150,
    'font.size': 11,
    'axes.grid': True,
    'grid.alpha': 0.35,
    'lines.linewidth': 1.8,
})

SERIES = {
    'gt':        {'label': 'Ground Truth',           'color': '#1f77b4'},
    'odo':       {'label': 'Odometria implementada', 'color': '#ff7f0e'},
    'dr':        {'label': 'Dead Reckoning',         'color': '#2ca02c'},
    'odo_husky': {'label': 'Odometria Husky',        'color': '#d62728'},
}

def carregar_csv(nome):
    caminho = PASTA_DADOS / nome
    if not caminho.exists():
        raise FileNotFoundError(f'Arquivo não encontrado: {caminho}')
    return pd.read_csv(caminho, comment='#')

def normalizar(df):
    """Translada para (0,0) e rotaciona para theta_0=0."""
    d = df.copy()
    x0     = d['field.x'].iloc[0]
    y0     = d['field.y'].iloc[0]
    theta0 = d['field.z'].iloc[0]

    # Translada
    dx = d['field.x'] - x0
    dy = d['field.y'] - y0

    # Rotaciona pelo ângulo inicial (alinha com eixo x)
    c, s = np.cos(-theta0), np.sin(-theta0)
    d['x']     = c * dx - s * dy
    d['y']     = s * dx + c * dy
    d['theta'] = d['field.z'] - theta0

    # Tempo em segundos
    d['t'] = (d['%time'] - d['%time'].iloc[0]) * 1e-9
    return d

print(f'Lendo dados de: {PASTA_DADOS}')

In [ ]:
dados = {}
for nome in SERIES:
    raw = carregar_csv(f'{nome}.csv')
    dados[nome] = normalizar(raw)
    d = dados[nome]
    print(f'{nome:12s}: {len(raw):>5} amostras | '
          f'duração {d["t"].iloc[-1]:.1f}s | '
          f'x_final={d["x"].iloc[-1]:.2f}m  '
          f'y_final={d["y"].iloc[-1]:.2f}m  '
          f'theta0_original={raw["field.z"].iloc[0]:.3f}rad')

## Trajetória XY

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

for nome, d in dados.items():
    s = SERIES[nome]
    ax.plot(d['x'], d['y'], label=s['label'], color=s['color'])
    ax.plot(d['x'].iloc[0],  d['y'].iloc[0],  'o', color=s['color'], ms=7)
    ax.plot(d['x'].iloc[-1], d['y'].iloc[-1], 'x', color=s['color'], ms=9, mew=2)

ax.set_xlabel('x [m]')
ax.set_ylabel('y [m]')
ax.set_title('Comparação das trajetórias\n(origem e orientação inicial alinhadas)')
ax.legend()
ax.axis('equal')
plt.tight_layout()
plt.savefig(PASTA_DADOS / 'plot_trajetoria_xy.png', dpi=150)
plt.show()

## Posição X, Y e orientação θ ao longo do tempo

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 11), sharex=True)

campos = [
    ('x',     'x relativo [m]',   'Posição X'),
    ('y',     'y relativo [m]',   'Posição Y'),
    ('theta', 'θ relativo [rad]', 'Orientação θ'),
]

for ax, (campo, ylabel, titulo) in zip(axes, campos):
    for nome, d in dados.items():
        s = SERIES[nome]
        ax.plot(d['t'], d[campo], label=s['label'], color=s['color'])
    ax.set_ylabel(ylabel)
    ax.set_title(titulo)
    ax.legend(loc='upper left', fontsize=9)

axes[-1].set_xlabel('Tempo [s]')
fig.suptitle('Comparação temporal — origem e orientação inicial alinhadas', y=1.01)
plt.tight_layout()
plt.savefig(PASTA_DADOS / 'plot_temporal_xytheta.png', dpi=150)
plt.show()

## Erro em relação ao Ground Truth

In [ ]:
gt = dados['gt']
fig, axes = plt.subplots(3, 1, figsize=(10, 11), sharex=True)
campos = [('x', 'Erro em x [m]'), ('y', 'Erro em y [m]'), ('theta', 'Erro em θ [rad]')]

for ax, (campo, ylabel) in zip(axes, campos):
    for nome in ['odo', 'dr', 'odo_husky']:
        d = dados[nome]
        s = SERIES[nome]
        gt_interp = np.interp(d['t'], gt['t'], gt[campo])
        erro = d[campo].values - gt_interp
        ax.plot(d['t'], erro, label=s['label'], color=s['color'])
    ax.axhline(0, color='gray', lw=0.8, ls='--')
    ax.set_ylabel(ylabel)
    ax.legend(loc='upper left', fontsize=9)

axes[-1].set_xlabel('Tempo [s]')
fig.suptitle('Erro em relação ao Ground Truth', y=1.01)
plt.tight_layout()
plt.savefig(PASTA_DADOS / 'plot_erro_gt.png', dpi=150)
plt.show()